In [1]:
import sys
sys.path.append("..")   # szukaj utils.py poziom wyżej

from utils import *
df = pd.read_parquet("../data/raw_data.parquet")

/home/akijaczek/Developer/Sentiment-analysis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/akijaczek/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to /home/akijaczek/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/akijaczek/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/akijaczek/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/akijaczek/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


---
## Preprocessing
Defined two text preprocessing functions used for cleaning and preparing raw text data for diffrent tasks.

- clean_minimal(text) performs basic cleaning by removing HTML tags, URLs, and extra whitespace, keeping the text largely unchanged otherwise.
- clean_full(text) applies a more advanced NLP pipeline: it removes HTML and URLs, expands contractions, filters out special characters and numbers, converts text to lowercase, tokenizes it, removes stopwords (while preserving important negations like “not” or “never”), lemmatizes words to their base form, and removes very short tokens.

In [2]:
def clean_minimal(text):
    text = re.sub(r'<[^>]+>', ' ', text) # Remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text) # Remove URLs
    text = re.sub(r'\s+', ' ', text).strip() # Normalize whitespace
    return text

def clean_full(text):
    text = re.sub(r'<[^>]+>', ' ', text) # Remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text) # Remove URLs
    text = contractions.fix(text)  # Expand contractions
    text = re.sub(r'[^a-zA-Z\s]', ' ', text) # Remove special characters and numbers 
    text = text.lower() # Lowercase
    tokens = word_tokenize(text) # Tokenize
    stop_words = set(stopwords.words('english')) # Remove stopwords
    stop_words -= {'not', 'no', 'nor', 'never', "don't", "doesn't", "didn't",
              "won't", "wouldn't", "can't", "couldn't", "shouldn't",
              "isn't", "aren't", "wasn't", "weren't"}
    tokens = [word for word in tokens if word not in stop_words]
    lemmatizer = WordNetLemmatizer() # Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    tokens = [word for word in tokens if len(word) > 1] # Remove short tokens (length < 2)
    return ' '.join(tokens)

In [3]:
df['text_raw'] = df['text'].apply(clean_minimal)
df['text_clean'] = df['text'].apply(clean_full)
    
df['text_length_original'] = df['text'].apply(len)
df['text_length_raw'] = df['text_raw'].apply(len)
df['text_length_clean'] = df['text_clean'].apply(len)
df['word_count_clean'] = df['text_clean'].apply(lambda x: len(x.split()))

In [4]:
print('\n---------- Example: ---------')
sample = df.iloc[10]
print(f"\nORIGINAL:\n{sample['text']}...")
print(f"\nTEXT_RAW:\n{sample['text_raw']}...")
print(f"\nTEXT_CLEAN:\n{sample['text_clean']}...")

print('\n---------- Statistics: ----------')
display(df[['text_length_original', 'text_length_raw',
            'text_length_clean', 'word_count_clean']].describe().round(2))

df.head()


---------- Example: ---------

ORIGINAL:
When I first read Armistead Maupins story I was taken in by the human drama displayed by Gabriel No one and those he cares about and loves. That being said, we have now been given the film version of an excellent story and are expected to see past the gloss of Hollywood...<br /><br />Writer Armistead Maupin and director Patrick Stettner have truly succeeded! <br /><br />With just the right amount of restraint Robin Williams captures the fragile essence of Gabriel and lets us see his struggle with issues of trust both in his personnel life(Jess) and the world around him(Donna).<br /><br />As we are introduced to the players in this drama we are reminded that nothing is ever as it seems and that the smallest event can change our lives irrevocably. The request to review a book written by a young man turns into a life changing event that helps Gabriel find the strength within himself to carry on and move forward.<br /><br />It's to bad that most pe

,text_length_original,text_length_raw,text_length_clean,word_count_clean
count,50000.00,50000.00,50000.00,50000.00
mean,1309.43,1286.76,812.17,120.83
std,989.73,972.37,627.25,90.98
min,32.00,32.00,17.00,3.00
25%,699.00,690.00,425.00,65.00
50%,970.00,954.00,597.00,90.00
75%,1590.25,1561.00,987.00,147.00
max,13704.00,13593.00,9114.00,1411.00


,review_id,split,rating,text,sentiment,sentiment_label,movie_id,text_raw,text_clean,text_length_original,text_length_raw,text_length_clean,word_count_clean
0,0,train,9,Bromwell High is a cartoon comedy. It ran at t...,pos,1,None,Bromwell High is a cartoon comedy. It ran at t...,bromwell high cartoon comedy ran time program ...,806,806,493,70
1,10000,train,8,Homelessness (or Houselessness as George Carli...,pos,1,None,Homelessness (or Houselessness as George Carli...,homelessness houselessness george carlin state...,2366,2322,1378,208
2,10001,train,10,Brilliant over-acting by Lesley Ann Warren. Be...,pos,1,None,Brilliant over-acting by Lesley Ann Warren. Be...,brilliant acting lesley ann warren best dramat...,841,841,558,84
3,10002,train,7,This is easily the most underrated film inn th...,pos,1,None,This is easily the most underrated film inn th...,easily underrated film inn brook cannon sure f...,663,663,422,66
4,10003,train,8,This is not the typical Mel Brooks film. It wa...,pos,1,None,This is not the typical Mel Brooks film. It wa...,not typical mel brook film much less slapstick...,647,647,349,54


In [5]:
df.to_parquet("../data/procerssed_data.parquet")
print("Preprocessed data saved in", os.listdir("../data"))

Preprocessed data saved in ['raw_data.parquet', 'procerssed_data.parquet']
